# ARGUS — Phase 2: train the heterogeneous GNN

Runs on **Kaggle** (GPU T4/P100) or **Colab** (GPU runtime).

**Kaggle:** *Settings → Accelerator → GPU*, *Internet → On*, and *Add Input →* dataset
`ealtman2019/ibm-transactions-for-aml`. Then *Run All*.

**Colab:** *Runtime → Change runtime type → GPU*, then run the Colab data cell instead of the Kaggle one.

In [ ]:
!git clone -q https://github.com/charansai-1411/ARGUS.git || (cd ARGUS && git pull -q)
%cd ARGUS

## Install PyG (+ pyg-lib for fast neighbour sampling)

In [ ]:
import torch
major, minor = torch.__version__.split('+')[0].split('.')[:2]
cuda = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheels = f'https://data.pyg.org/whl/torch-{major}.{minor}.0+{cuda}.html'
print('torch', torch.__version__, '| wheels:', wheels)
!pip install -q torch_geometric lightgbm pyarrow pyyaml
!pip install -q pyg_lib -f {wheels} || pip install -q torch_sparse -f {wheels}
!pip install -q -e . --no-deps

## Data location
Run **one** of the two cells below.

In [ ]:
# Kaggle: the dataset is attached as an input
RAW = '/kaggle/input/ibm-transactions-for-aml'
WORK = '/kaggle/working'

In [ ]:
# Colab: download with the Kaggle API (needs your kaggle.json from kaggle.com -> Settings -> API)
# from google.colab import files; files.upload()
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# for f in ['HI-Small_Trans.csv', 'HI-Small_Patterns.txt', 'HI-Small_accounts.csv']:
#     !kaggle datasets download -d ealtman2019/ibm-transactions-for-aml -f {f} -p /content/raw --unzip
# RAW, WORK = '/content/raw', '/content'

In [ ]:
import os
for f in ['HI-Small_Trans.csv', 'HI-Small_Patterns.txt', 'HI-Small_accounts.csv']:
    print(f, os.path.exists(f'{RAW}/{f}'))

## Quick check (small slice, 1 epoch) — should finish in a few minutes

In [ ]:
!python scripts/train_gnn.py --raw-dir {RAW} --processed-dir {WORK}/processed --output-dir {WORK}/outputs     --limit 300000 --epochs 1

## Full run on HI-Small
Try `--conv gine`, `--conv gat` and `--conv sage` and compare with the LightGBM baseline (test PR-AUC 0.296, F1 0.367).

In [ ]:
!python scripts/train_gnn.py --raw-dir {RAW} --processed-dir {WORK}/processed --output-dir {WORK}/outputs --conv gine

In [ ]:
import json, glob
for f in sorted(glob.glob(f'{WORK}/outputs/gnn_*.json')):
    r = json.load(open(f))
    print(f"{os.path.basename(f):40s} PR-AUC {r['pr_auc']:.4f}  F1 {r['f1']:.4f}  "
          f"P {r['precision']:.3f}  R {r['recall']:.3f}  ring@1 {r['ring_recall@1']:.3f}  ring@.5 {r['ring_recall@0.5']:.3f}")

Download `outputs/*.json` (and the `.pt` checkpoint) from the output panel and commit the JSON results to `results/` in the repo.